# CardioRisk Studio — prédiction de maladie cardiaque

**Notebook pédagogique réorganisé** · Classification binaire à partir du dataset UCI Heart Disease.

**Objectif :** prédire la cible `target` : `0 = absence` et `1 = présence` de maladie (diagnostic d’origine `num > 0`). Ce notebook montre les étapes dans leur ordre logique : comprendre les données, explorer leur qualité, corriger les valeurs invalides, explorer les relations, puis préparer et évaluer des modèles.

> **Important :** ce travail est pédagogique. Le dataset ancien et la validation sur un seul découpage ne permettent pas un usage médical ni une conclusion clinique.

## 1. Importer les bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Charger le fichier et vérifier sa structure

Le CSV doit se trouver dans le même dossier que ce notebook. Garder le dataset brut dans `df_raw` permet de revenir aux données originales sans perdre d’information.

In [ ]:
DATA_PATH = "heart_disease_uci.csv"
df_raw = pd.read_csv(DATA_PATH)
print(f"Dimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

## 3. Comprendre les colonnes avant le nettoyage

Les variables catégorielles décrivent des groupes ou des modalités : on ne calcule pas une moyenne de leurs étiquettes. Les variables numériques représentent des mesures ou des quantités. Le type informatique (`int`, `float`, `object`) ne suffit pas toujours à déterminer le sens statistique : par exemple `ca` est enregistré comme nombre, mais représente un décompte de vaisseaux.

In [ ]:
dictionnaire = pd.DataFrame([
    ["id", "Identifiant de ligne", "Métadonnée — exclue"],
    ["age", "Âge en années", "Numérique"],
    ["sex", "Sexe enregistré (Male/Female)", "Catégorielle"],
    ["dataset", "Centre d'origine", "Métadonnée — exclue"],
    ["cp", "Type de douleur thoracique", "Catégorielle"],
    ["trestbps", "Pression artérielle au repos (mm Hg)", "Numérique"],
    ["chol", "Cholestérol sérique (mg/dl)", "Numérique"],
    ["fbs", "Glycémie à jeun > 120 mg/dl", "Catégorielle binaire"],
    ["restecg", "Résultat ECG au repos", "Catégorielle"],
    ["thalch", "Fréquence cardiaque maximale à l'effort", "Numérique"],
    ["exang", "Angine provoquée par l'effort", "Catégorielle binaire"],
    ["oldpeak", "Dépression ST à l'effort", "Numérique"],
    ["slope", "Pente du segment ST à l'effort", "Catégorielle ordinale"],
    ["ca", "Nombre de vaisseaux majeurs observés (0–3)", "Numérique / décompte"],
    ["thal", "Résultat du test thal/perfusion", "Catégorielle"],
    ["num", "Diagnostic d'origine (0–4)", "Cible d'origine"],
], columns=["Colonne", "Signification", "Nature / rôle"])
dictionnaire

### Modalités catégorielles en langage simple

- `cp` : `typical angina` (angine typique), `atypical angina` (angine atypique), `non-anginal` (douleur non angineuse), `asymptomatic` (sans douleur rapportée).
- `sex` : sexe enregistré (`Male`, `Female`).
- `fbs` : glycémie à jeun supérieure à 120 mg/dl (`True/False`).
- `restecg` : résultat de l’ECG au repos (`normal`, `lv hypertrophy`, `st-t abnormality`).
- `exang` : douleur angineuse provoquée par l’effort (`True/False`).
- `slope` : pente ST (`upsloping`, `flat`, `downsloping`).
- `thal` : résultat de perfusion (`normal`, `fixed defect`, `reversable defect`).

La cible binaire du projet sera créée à partir de `num` : les valeurs `1` à `4` sont regroupées en présence de maladie. Cette simplification fait perdre l’information de sévérité.

## 4. Explorer les valeurs manquantes et la cible dans les données brutes

In [ ]:
# Nombre et pourcentage de valeurs manquantes (NaN) dans chaque colonne
manquants = pd.DataFrame({
    "Nombre manquant": df_raw.isna().sum(),
    "Pourcentage (%)": (df_raw.isna().mean() * 100).round(1)
}).sort_values("Nombre manquant", ascending=False)
manquants[manquants["Nombre manquant"] > 0]

In [ ]:
print("Doublons complets :", df_raw.duplicated().sum())
print("Valeurs de num :")
print(df_raw["num"].value_counts(dropna=False).sort_index())
print("\nEffectifs après regroupement binaire :")
print((df_raw["num"] > 0).value_counts().rename(index={False: "Absence", True: "Présence"}))

**À retenir :** dans ce fichier, `cp` n’a pas de valeur manquante. Les absences sont concentrées surtout dans `ca`, `thal` et `slope`. Les `NaN` sont des données non renseignées; ils ne signifient pas zéro.

## 5. Nettoyage minimal : corriger les zéros invalides et créer la cible

Un zéro peut être une vraie mesure pour certaines variables, mais ici le dictionnaire du projet considère `chol == 0` et `trestbps == 0` comme des mesures invalides. On les transforme donc en `NaN` afin de les traiter comme inconnues, et non comme des mesures réellement nulles.

**Ce n’est pas encore l’imputation par médiane.** À cette étape, on ne remplace rien par une médiane : on marque les valeurs invalides. L’imputation statistique sera apprise plus tard uniquement à partir de l’ensemble d’entraînement, dans le pipeline du modèle.

In [ ]:
data = df_raw.copy()
print("Zéros détectés avant correction :")
print("chol     :", (data["chol"] == 0).sum())
print("trestbps :", (data["trestbps"] == 0).sum())

data["chol"] = data["chol"].replace(0, np.nan)
data["trestbps"] = data["trestbps"].replace(0, np.nan)
data["target"] = (data["num"] > 0).astype(int)

print("\nCible créée : 0 = absence, 1 = présence")
print(data["target"].value_counts().sort_index())

In [ ]:
# Comparer avant et après correction : des valeurs manquantes en plus sont attendues
comparaison = pd.DataFrame({
    "manquants avant": df_raw.isna().sum(),
    "manquants après correction": data.isna().sum()
})
comparaison[comparaison.max(axis=1) > 0].sort_values("manquants après correction", ascending=False)

## 6. Exploration visuelle après correction

Ces graphiques servent à décrire le dataset et à repérer des différences entre groupes. Ils ne démontrent ni cause ni capacité de diagnostic.

In [ ]:
# Répartition de la cible
ax = data["target"].value_counts().sort_index().rename(index={0: "Absence", 1: "Présence"}).plot(
    kind="bar", color=["#4C78A8", "#F58518"], figsize=(6, 4)
)
ax.set_title("Répartition de la cible")
ax.set_xlabel("Maladie cardiaque")
ax.set_ylabel("Nombre de dossiers")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Distribution de l'âge selon la cible
plt.figure(figsize=(7, 4))
sns.histplot(data=data, x="age", hue="target", bins=20, kde=True,
             palette={0: "#4C78A8", 1: "#F58518"})
plt.title("Âge selon la cible (0 = absence, 1 = présence)")
plt.xlabel("Âge (années)")
plt.ylabel("Effectif")
plt.show()

In [ ]:
# Comparer les variables d'effort
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=data, x="target", y="thalch", hue="target", legend=False, ax=axes[0])
axes[0].set_title("Fréquence cardiaque maximale")
axes[0].set_xlabel("Cible (0/1)")
axes[0].set_ylabel("Battements/minute")
sns.boxplot(data=data, x="target", y="oldpeak", hue="target", legend=False, ax=axes[1])
axes[1].set_title("Dépression ST (oldpeak)")
axes[1].set_xlabel("Cible (0/1)")
plt.tight_layout()
plt.show()

Les boîtes et distributions montrent des associations observées dans cet échantillon. Elles ne signifient pas que l’âge ou une mesure donnée cause la maladie.

In [ ]:
# Type de douleur thoracique (cp) selon la cible
plt.figure(figsize=(9, 4))
sns.countplot(data=data, x="cp", hue="target", hue_order=[0, 1],
              palette={0: "#4C78A8", 1: "#F58518"})
plt.title("Type de douleur thoracique selon la cible")
plt.xlabel("Type de douleur (cp)")
plt.ylabel("Nombre de dossiers")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Cible", labels=["0 = absence", "1 = présence"])
plt.tight_layout()
plt.show()

In [ ]:
# Visualiser les colonnes incomplètes
miss = data.isna().sum().sort_values()
miss = miss[miss > 0]
plt.figure(figsize=(7, 4))
miss.plot(kind="barh", color="#E45756")
plt.title("Valeurs manquantes après correction des zéros invalides")
plt.xlabel("Nombre de valeurs manquantes")
plt.ylabel("Variable")
plt.show()

## 7. Étudier les associations sans encoder les catégories comme des nombres

Une corrélation de Pearson convient surtout aux variables numériques et mesure une association linéaire. Il ne faut pas convertir arbitrairement les catégories (`cp`, `thal`, etc.) en 0, 1, 2… puis interpréter cette corrélation : le résultat dépendrait de l’ordre choisi pour les étiquettes. Ici, on calcule donc la matrice numérique seulement, en excluant l’identifiant, la cible d’origine et la métadonnée du centre. Les manquants sont ignorés paire par paire pour cette exploration.

In [ ]:
numeric_for_corr = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca", "target"]
corr = data[numeric_for_corr].corr(method="pearson")
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélations de Pearson — variables numériques seulement")
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation numérique avec la cible (descriptive, pas causale)
corr["target"].drop("target").sort_values(key=lambda s: s.abs(), ascending=False)

Pour `cp` et les autres catégories, le graphique de répartition par cible est plus approprié qu’une corrélation calculée après un codage arbitraire. Une analyse statistique catégorielle avancée (par exemple chi-deux / V de Cramér) peut être ajoutée plus tard, mais n’est pas nécessaire pour comprendre et exécuter ce premier notebook.

## 8. Définir les entrées du modèle et séparer train/test

On retire `id` (identifiant), `dataset` (centre utilisé ici comme métadonnée) et `num` (diagnostic d’origine, qui a servi à créer `target`). Le jeu de test est mis de côté avant d’apprendre les médianes, les modes, les échelles et les catégories du modèle.

In [ ]:
numeric_features = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
features = numeric_features + categorical_features

X = data[features].copy()
y = data["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Entraînement :", X_train.shape, " Test :", X_test.shape)
print("Proportion cible dans train :\n", y_train.value_counts(normalize=True).sort_index().round(3))
print("Proportion cible dans test :\n", y_test.value_counts(normalize=True).sort_index().round(3))

## 9. Préparer les colonnes dans un pipeline

**Pourquoi la médiane ?** Pour une mesure numérique, la médiane est moins sensible aux valeurs extrêmes que la moyenne. **Pourquoi le mode ?** Pour une catégorie, on remplace une absence par la modalité la plus fréquente plutôt que de calculer une moyenne sans sens.

Le pipeline ajuste ces valeurs **sur `X_train` seulement** et applique ensuite les mêmes transformations à `X_test`. Ainsi, le test ne participe pas à la préparation qui apprend le modèle. Les variables catégorielles sont converties en indicateurs 0/1 avec One-Hot Encoding. La standardisation est utile aux modèles sensibles aux échelles (notamment la régression logistique et KNN).

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
print("Préprocesseur défini : il sera ajusté pendant le fit sur X_train.")

## 10. Entraîner une baseline et trois modèles

La baseline prédit toujours la classe majoritaire. Elle sert de point de comparaison minimal. Les autres modèles sont entraînés avec le même prétraitement.

In [ ]:
models = {
    "Baseline (classe majoritaire)": DummyClassifier(strategy="most_frequent"),
    "Régression logistique": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "KNN (k=7)": KNeighborsClassifier(n_neighbors=7)
}

pipelines = {}
for name, estimator in models.items():
    pipe = Pipeline(steps=[("preparation", preprocessor), ("modele", estimator)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print(f"Modèle entraîné : {name}")

## 11. Évaluer les modèles sur le jeu de test

L’accuracy donne la part totale de réponses correctes. Le rappel mesure la part des cas positifs retrouvés. La précision mesure la part des prédictions positives qui sont correctes. Le F1 combine précision et rappel. L’AUC évalue le classement des probabilités sur plusieurs seuils. Ces scores restent des résultats exploratoires sur ce jeu de test.

In [ ]:
rows = []
predictions = {}
for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    predictions[name] = y_pred
    if hasattr(pipe, "predict_proba"):
        y_proba = pipe.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred.astype(float)
    rows.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Précision": precision_score(y_test, y_pred, zero_division=0),
        "Rappel": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results = pd.DataFrame(rows).set_index("Modèle").sort_values("F1", ascending=False)
results.round(3)

In [ ]:
ax = results[["Accuracy", "Précision", "Rappel", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1)
)
ax.set_title("Comparaison des modèles sur le jeu de test")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de confusion du meilleur modèle par F1 (hors baseline)
ml_results = results.drop(index="Baseline (classe majoritaire)", errors="ignore")
best_name = ml_results["F1"].idxmax()
best_model = pipelines[best_name]
print("Modèle choisi selon F1 :", best_name)
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["Absence (0)", "Présence (1)"], zero_division=0))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=["Absence (0)", "Présence (1)"], cmap="Blues"
)
plt.title(f"Matrice de confusion — {best_name}")
plt.show()

## 12. Conclusion et limites

1. La cible a été construite en regroupant `num = 1, 2, 3, 4` en présence de maladie (`target = 1`); `num = 0` devient absence (`target = 0`).
2. Les zéros de `chol` et `trestbps` ont été traités comme invalides puis marqués manquants. L’imputation médiane/mode intervient **après le split**, dans le pipeline ajusté sur l’entraînement.
3. La corrélation de Pearson a été limitée aux variables numériques; les catégories sont explorées avec leurs effectifs plutôt qu’avec des codes numériques arbitraires.
4. Les performances peuvent varier selon le découpage. Une validation croisée et une validation sur un autre jeu de données seraient nécessaires pour une étude plus robuste.
5. Ce dataset historique de 920 observations est incomplet et multi-centres. Les scores ne sont pas une preuve de validité clinique. **Ne pas utiliser ce modèle pour poser un diagnostic ou décider d’un traitement.**